# RoadSafe Delhi — Random Forest Model Development
## Assigned Member: Nakshtr
### Task: Non-Parametric Ensemble Tree Classification & Feature Importance

---

## 1. Project Introduction & Assigned Member
- **Project:** RoadSafe Delhi — Explainable Road Risk Mapping and Safety Intervention Prioritization Using Machine Learning
- **Assigned Member:** Nakshtr
- **Model Assignment:** Random Forest Classifier (`sklearn.ensemble.RandomForestClassifier`)
- **Role:** Data cleaning, ensemble model development, hyperparameter tuning, report maintenance, and Gini feature importance analysis.


## 2. Problem Statement & Model Objective
The objective is to train a non-parametric Random Forest ensemble to classify road segments as hazardous (`is_crash_segment = 1`) or baseline (`is_crash_segment = 0`).
- **Ensemble Rationale:** Handles non-linear feature interactions, resistant to overfitting, and provides intrinsic Gini impurity feature importances for municipal explainability.


## 3. Dataset Loading from Common Approved Dataset
Loading the shared spatial partitions from `data_final/`.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../..")) if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
DATA_FINAL_DIR = os.path.join(PROJECT_ROOT, "data_final")
ARTIFACTS_DIR = os.path.join(PROJECT_ROOT, "models", "random_forest", "artifacts")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "models", "random_forest", "outputs")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

df_train = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "train.parquet"))
df_val = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "validation.parquet"))
df_test = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "test.parquet"))

print(f"Loaded Train: {len(df_train):,}, Val: {len(df_val):,}, Test: {len(df_test):,}")


## 4. Target Variable Selection
Selected Target: `is_crash_segment` (Binary Hazard Occurrence).


In [ ]:
target_col = "is_crash_segment"
print(f"Target Variable: {target_col}")
print(f"Train positive rate: {df_train[target_col].mean()*100:.2f}%")
print(f"Val positive rate:   {df_val[target_col].mean()*100:.2f}%")
print(f"Test positive rate:  {df_test[target_col].mean()*100:.2f}%")


## 5. Approved Feature Selection
Loading `feature_contract.json` and asserting that zero quarantined columns enter the feature matrix.


In [ ]:
with open(os.path.join(DATA_FINAL_DIR, "feature_contract.json"), "r") as f:
    contract = json.load(f)

num_features = contract["approved_numerical_predictors"]
cat_features = contract["approved_categorical_predictors"]
approved_predictors = num_features + cat_features

forbidden = set(contract["quarantined_leakage_columns"]).union(set(contract["target_definitions"].keys()))
assert len(set(approved_predictors).intersection(forbidden)) == 0, "FATAL: Leakage detected!"
print("Approved Predictors:", approved_predictors)


## 6. Data Preprocessing
Encoding categorical features (`type`) using OneHotEncoder fitted strictly on training data.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(df_train[cat_features])

X_train_num = df_train[num_features].values
X_val_num = df_val[num_features].values
X_test_num = df_test[num_features].values

X_train_cat = encoder.transform(df_train[cat_features])
X_val_cat = encoder.transform(df_val[cat_features])
X_test_cat = encoder.transform(df_test[cat_features])

X_train = np.hstack([X_train_num, X_train_cat])
X_val = np.hstack([X_val_num, X_val_cat])
X_test = np.hstack([X_test_num, X_test_cat])

y_train = df_train[target_col].values
y_val = df_val[target_col].values
y_test = df_test[target_col].values

feature_names = num_features + encoder.get_feature_names_out(cat_features).tolist()
print(f"Prepared {X_train.shape[1]} features for Random Forest.")


## 7. Train-Validation-Test Data Verification (Common Spatial Split)
Verifying disjoint spatial block boundaries.


In [ ]:
print("Train Blocks:", df_train['spatial_block'].unique().tolist())
print("Val Blocks:  ", df_val['spatial_block'].unique().tolist())
print("Test Blocks: ", df_test['spatial_block'].unique().tolist())
assert set(df_train['spatial_block']).isdisjoint(set(df_val['spatial_block']))
assert set(df_train['spatial_block']).isdisjoint(set(df_test['spatial_block']))
print("Spatial Integrity Confirmed.")


## 8. Model Architecture & Configuration
Configuring `RandomForestClassifier` with balanced class weights and tree parameters.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
print("Configured Random Forest Model:", rf_model)


## 9. Model Training
Training Random Forest strictly on `X_train`, `y_train`.


In [ ]:
rf_model.fit(X_train, y_train)
print("Random Forest Training Completed.")


## 10. Validation & Hyperparameter Tuning
Evaluating on the validation set and tuning decision threshold to maximize F1-score.


In [ ]:
from sklearn.metrics import precision_recall_curve

y_val_prob = rf_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)
f1_scores = np.zeros_like(thresholds)
for i, t in enumerate(thresholds):
    f1_scores[i] = f1_score(y_val, (y_val_prob >= t).astype(int), zero_division=0)

best_idx = np.argmax(f1_scores)
optimal_threshold = float(thresholds[best_idx])
print(f"Optimal Validation Threshold: {optimal_threshold:.4f} (Validation F1 = {f1_scores[best_idx]:.4f})")


## 11. Final Testing on the Untouched Test Dataset
Applying the frozen validation threshold unchanged to `X_test`.


In [ ]:
y_test_prob = rf_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= optimal_threshold).astype(int)

print(f"Test Accuracy:  {accuracy_score(y_test, y_test_pred)*100:.2f}%")
print(f"Test Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test Recall:    {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test F1-Score:  {f1_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"Test PR-AUC:    {average_precision_score(y_test, y_test_prob):.4f}")


## 12. Evaluation Metrics & Visualizations
Plotting Feature Importances and Confusion Matrix.


In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", ax=ax)
ax.set_title(f"Random Forest Test Confusion Matrix (Threshold={optimal_threshold:.3f})")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "confusion_matrix.png"), dpi=200)
plt.show()

# Feature Importances
importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
importances.head(12).plot(kind="barh", color="#27ae60")
plt.title("Random Forest Feature Importances (MDI / Gini)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "feature_importance.png"), dpi=200)
plt.show()


## 13. Results Interpretation
Discussing top contributing features (e.g. carriage lanes, builtup density, length) and municipal intervention takeaways.


In [ ]:
print("Top 5 Driving Risk Factors:\n", importances.head(5))


## 14. Model and Artifact Saving
Saving model, encoder, threshold, and metrics.


In [ ]:
import joblib

joblib.dump(rf_model, os.path.join(ARTIFACTS_DIR, "random_forest_model.joblib"))
joblib.dump(encoder, os.path.join(ARTIFACTS_DIR, "encoder.joblib"))
with open(os.path.join(ARTIFACTS_DIR, "threshold.json"), "w") as f:
    json.dump({"optimal_threshold": optimal_threshold}, f, indent=2)

print(f"Saved Random Forest artifacts to {ARTIFACTS_DIR}")


## 15. Final Summary & Observations
Summary of model strengths, tree depth trade-offs, and comparison with baseline.
